In [1]:
import pandas as pd

correct_columns = [
    'app_id', 'Name', 'Release date', 'Estimated owners', 'Peak CCU',
    'Required age', 'Price', 'Discount', 'DLC count', 'About the game',
    'Supported languages', 'Full audio languages', 'Reviews', 'Header image',
    'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux',
    'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative',
    'Score rank', 'Achievements', 'Recommendations', 'Notes',
    'Average playtime forever', 'Average playtime two weeks',
    'Median playtime forever', 'Median playtime two weeks',
    'Developers', 'Publishers', 'Categories', 'Genres', 'Tags',
    'Screenshots', 'Movies'
]

gamesgenres = pd.read_csv(
    "../data/games_inc.csv",
    header=0,          # 기존 헤더 줄은 버리고
    names=correct_columns  # 이 이름들을 강제로 매핑
)

print(gamesgenres.shape)  # (125855, 40) 이 나오면 정상
recommendations = pd.read_csv("../data/recommendations.csv")
print(recommendations.shape)
games_recs_merged = pd.merge(gamesgenres, recommendations, on='app_id', how='inner')

print(games_recs_merged.shape)
games_recs_merged.head()
games_recs_merged.to_csv(
    "data/games_recs_merged.csv",
    index=False,
    encoding="utf-8-sig"
)

(125855, 40)
(41154794, 8)
(40230954, 47)


OSError: Cannot save file into a non-existent directory: 'data'

In [15]:
user_review_counts = games_recs_merged.groupby('user_id').size().reset_index(name='review_count')
user_review_counts = user_review_counts.sort_values('review_count', ascending=False)

print(user_review_counts.shape)
user_review_counts.head(10)

(13589197, 2)


,user_id,review_count
11152009,11764552,5826
4720056,5112758,3933
11046188,11656130,3758
5261744,5669734,3366
10946125,11553593,3209
4990278,5390510,2853
4149867,4457971,2664
4018422,4318160,2647
545448,574944,2587
1293097,1365585,2327


In [7]:
print(user_review_counts['review_count'].describe())
print(user_review_counts['review_count'].quantile([0.5, 0.9, 0.95, 0.99, 0.999]))

count    1.358920e+07
mean     2.960510e+00
std      7.959970e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      3.000000e+00
max      5.826000e+03
Name: review_count, dtype: float64
0.500     1.0
0.900     6.0
0.950     9.0
0.990    25.0
0.999    78.0
Name: review_count, dtype: float64


In [11]:
lower_bound = 10
upper_bound = 78  # 99.9 percentile 근사값 (봇/이상치 배제)

eligible_users = user_review_counts[
    (user_review_counts['review_count'] >= lower_bound) &
    (user_review_counts['review_count'] <= upper_bound)
]
print(f"평가 대상 유저 수: {len(eligible_users):,}")

평가 대상 유저 수: 647,035


In [13]:
eval_df = games_recs_merged[
    games_recs_merged["user_id"].isin(eligible_users["user_id"])
]

In [22]:
import pandas as pd

pd.set_option("display.max_columns", None)  # 모든 컬럼 표시
pd.set_option("display.max_colwidth", None) # 긴 문자열도 생략하지 않음
pd.set_option("display.width", None)        # 화면 너비 제한 해제

games_recs_merged.head()

print(eval_df.head(7))

    app_id                                   Name  Release date  \
1   496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016   
2   496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016   
4   496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016   
5   496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016   
10  496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016   
11  496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016   
12  496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016   

   Estimated owners  Peak CCU  Required age  Price  Discount  DLC count  \
1         0 - 20000         0             0   5.24        65          0   
2         0 - 20000         0             0   5.24        65          0   
4         0 - 20000         0             0   5.24        65          0   
5         0 - 20000         0             0   5.24        65          0   
10        0 - 20000         0             0   5.24        65          0   
11        0 -

In [23]:
eval_df[
    [
        "user_id",
        "app_id",
        "Name",
        "Genres",
        "Tags",
        "is_recommended",
        "hours"
    ]
].head(10)

,user_id,app_id,Name,Genres,Tags,is_recommended,hours
1,7177628,496350,Supipara - Chapter 1 Spring Has Come!,Adventure,"Adventure,Visual Novel,Anime,Cute",True,25.0
2,10659614,496350,Supipara - Chapter 1 Spring Has Come!,Adventure,"Adventure,Visual Novel,Anime,Cute",True,4.0
4,8508352,496350,Supipara - Chapter 1 Spring Has Come!,Adventure,"Adventure,Visual Novel,Anime,Cute",True,10.0
5,7112108,496350,Supipara - Chapter 1 Spring Has Come!,Adventure,"Adventure,Visual Novel,Anime,Cute",True,46.0
10,10349601,496350,Supipara - Chapter 1 Spring Has Come!,Adventure,"Adventure,Visual Novel,Anime,Cute",True,0.0
11,6066119,496350,Supipara - Chapter 1 Spring Has Come!,Adventure,"Adventure,Visual Novel,Anime,Cute",True,2.0
12,5944835,496350,Supipara - Chapter 1 Spring Has Come!,Adventure,"Adventure,Visual Novel,Anime,Cute",True,505.0
13,4450098,496350,Supipara - Chapter 1 Spring Has Come!,Adventure,"Adventure,Visual Novel,Anime,Cute",True,12.0
14,14988,496350,Supipara - Chapter 1 Spring Has Come!,Adventure,"Adventure,Visual Novel,Anime,Cute",True,39.0
15,5552167,496350,Supipara - Chapter 1 Spring Has Come!,Adventure,"Adventure,Visual Novel,Anime,Cute",True,17.0


In [35]:
print(eval_df.shape)
print(eval_df["user_id"].nunique())

(12165200, 47)
647035


In [36]:
from sklearn.model_selection import train_test_split
import pandas as pd

train_list = []
test_list = []

# 사용자별 30%를 Test로 추출
test_df = (
    eval_df.groupby("user_id", group_keys=False)
           .sample(frac=0.3, random_state=42)
)

# 나머지를 Train으로 사용
train_df = eval_df.drop(test_df.index)

print(train_df.shape)
print(test_df.shape)

(8531170, 47)
(3634030, 47)


In [41]:
train_df.to_csv(
    r"C:\projects\Game-Recommendation-System\data\split\train.csv",
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    r"C:\projects\Game-Recommendation-System\data\split\test.csv",
    index=False,
    encoding="utf-8-sig"
)

In [37]:
print(eval_df.dtypes)

app_id                          int64
Name                              str
Release date                      str
Estimated owners                  str
Peak CCU                        int64
Required age                    int64
Price                         float64
Discount                        int64
DLC count                       int64
About the game                    str
Supported languages               str
Full audio languages              str
Reviews                           str
Header image                      str
Website                           str
Support url                       str
Support email                     str
Windows                          bool
Mac                              bool
Linux                            bool
Metacritic score                int64
Metacritic url                    str
User score                      int64
Positive                        int64
Negative                        int64
Score rank                    float64
Achievements

In [ ]:


for user_id, user_data in eval_df.groupby("user_id"):

    print(f"\n===== User {user_id} =====")
    print(user_data.reset_index(drop=True)[ 
        ["app_id", "Name", "is_recommended", "hours"]
    ])

    count += 1

    if count == 5:
        break


===== User 0 =====
           app_id                                                                               Name  is_recommended  hours
2477804    418950                                                      DreadOut: Keepers of The Dark           False    5.0
5339726   1454400                                                                     Cookie Clicker            True  762.6
5446718    357900                                                                     Make it indie!           False    1.0
8787485    222880                                                                         Insurgency            True  162.5
9983261    226960                                                                   Ironclad Tactics           False    1.0
12410109    63710                                                                    BIT.TRIP RUNNER           False    2.0
13287619   487370                                                                               Akin           F

In [3]:
print(recommendations.head())

    app_id  helpful  funny        date  is_recommended  hours  user_id  \
0   975370        0      0  2022-12-12            True   36.3    51580   
1   304390        4      0  2017-02-17           False   11.5     2586   
2  1085660        2      0  2019-11-17            True  336.5   253880   
3   703080        0      0  2022-09-23            True   27.4   259432   
4   526870        0      0  2021-01-10            True    7.9    23869   

   review_id  
0          0  
1          1  
2          2  
3          3  
4          4  


In [4]:
print(gamesgenres.head())

     AppID                                   Name  Release date  \
0  2539430             Black Dragon Mage Playtest   Aug 1, 2023   
1   496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016   
2  1034400      Mystery Solitaire The Black Raven   May 6, 2019   
3  3292190            버튜버 파라노이아 - Vtuber Paranoia  Oct 31, 2024   
4  3631080                          Maze Quest VR  Apr 24, 2025   

  Estimated owners  Peak CCU  Required age  Price  Discount  DLC count  \
0            0 - 0         0             0   0.00         0          0   
1        0 - 20000         0             0   5.24        65          0   
2        0 - 20000         0             0   4.99         0          0   
3        0 - 20000         1             0   8.99         0          1   
4        0 - 20000         0             0   4.99         0          0   

                                      About the game  ...  \
0                                                NaN  ...   
1  Springtime, April: when the